#### Load Bronze Incremental Table

In [0]:
bronze_df = spark.table("ecommerce.bronze.incremental_orders")
display(bronze_df)

#### Create Silver Table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce.silver.incremental_orders_silver (
    order_id INT,
    order_date DATE,
    customer_name STRING,
    product STRING,
    category STRING,
    quantity INT,
    price DOUBLE,
    ingested_at TIMESTAMP
)
USING DELTA
""")


#### Prepare Bronze Data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("bronze_ingested_at").desc(),
        F.col("source_file").desc()
    )
)

bronze_clean_df = (
    bronze_df
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "order_id",
        "order_date",
        "customer_name",
        "product",
        "category",
        "quantity",
        "price",
        "discount_percentage",
        "ingested_at"
    )
)

display(bronze_clean_df)

MERGE Bronze → Silver (UPSERT)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, "ecommerce.silver.incremental_orders_silver")

silver_table.alias("s").merge(
    bronze_clean_df.alias("b"),
    "s.order_id = b.order_id"
).withSchemaEvolution() \
 .whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

Load Silver Table

In [0]:
silver_df = spark.table("ecommerce.silver.incremental_orders_silver")
display(silver_df)

Record Count Check

In [0]:
print("Total Records in Silver Table:", silver_df.count())

In [0]:
display(silver_df.filter("order_id = 2002"))

In [0]:
display(silver_df.filter("discount_percentage IS NOT NULL"))

In [0]:
spark.sql("""
DESCRIBE HISTORY ecommerce.silver.incremental_orders_silver
""").show(truncate=False)